# VectorTA Computation Library EDA

VectorTA provides fast technical indicators with CPU and CUDA-oriented execution paths. This notebook documents the integration shape, validates the installed package, and shows how its outputs can feed feature engineering.

In [1]:
from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

def required_library_status(module_name: str) -> pd.DataFrame:
    try:
        module = importlib.import_module(module_name)
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": True,
            "version": getattr(module, "__version__", "unknown"),
            "module_file": getattr(module, "__file__", None),
        }])
    except Exception as exc:
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": False,
            "version": None,
            "module_file": None,
            "error": f"{type(exc).__name__}: {exc}",
        }])

def synthetic_prices(rows: int = 360, symbol: str = "AAPL") -> pd.DataFrame:
    rng = np.random.default_rng(7)
    dates = pd.date_range("2024-01-02", periods=rows, freq="B")
    drift = 0.00055
    shock = rng.normal(0, 0.015, rows)
    close = 100 * np.exp(np.cumsum(drift + shock))
    open_ = close * (1 + rng.normal(0, 0.003, rows))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.02, rows))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.02, rows))
    volume = rng.integers(900_000, 4_500_000, rows).astype(float)
    return pd.DataFrame({
        "symbol": symbol,
        "date": dates,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    })

prices = synthetic_prices()
prices.head()

,symbol,date,open,high,low,close,volume
0,AAPL,2024-01-02,100.2080,101.2355,98.8738,100.0569,"4,479,129.0000"
1,AAPL,2024-01-03,101.1259,102.2340,100.4497,100.5615,"2,365,734.0000"
2,AAPL,2024-01-04,100.3819,101.9042,99.3952,100.2040,"1,409,205.0000"
3,AAPL,2024-01-05,98.9452,99.8670,97.0239,98.9286,"4,225,356.0000"
4,AAPL,2024-01-08,97.8130,98.6722,97.4943,98.3103,"2,886,674.0000"


## Required Dependency Status

In [2]:
import importlib.metadata as md
import importlib.util
import vector_ta

vector_ta_status = pd.DataFrame([
    {
        "package": "vector-ta",
        "import_name": "vector_ta",
        "installed": True,
        "version": md.version("vector-ta"),
        "extension": importlib.util.find_spec("vector_ta.vector_ta").origin,
        "cuda_symbol_count": sum(1 for name in dir(vector_ta) if "cuda" in name.lower()),
        "sample_cuda_symbols": ", ".join([name for name in dir(vector_ta) if "cuda" in name.lower()][:5]),
    }
])
vector_ta_status


,package,import_name,installed,version,extension,cuda_symbol_count,sample_cuda_symbols
0,vector-ta,vector_ta,True,0.2.8,/home/jlee153232/miniconda3/lib/python3.13/sit...,402,"FramaCudaBatchPlan, MabCudaBatchPlan, MediumAd..."


## Required Feature Engineering Contract

`vector_ta` is a required computation-library dependency. Its outputs should be stored as a separate computation-library family, not mixed into pandas-ta-classic outputs. The contract should look like a `BuiltFeatureSet`: MultiIndex `(date, symbol)`, prefixed feature columns, and no target leakage.

In [3]:
# Required integration contract: VectorTA adapters should return the same shape as existing BuiltFeatureSet outputs.
from dataclasses import dataclass
import vector_ta

@dataclass(frozen=True)
class BuiltFeatureSet:
    family: str
    df: pd.DataFrame
    feature_cols: list[str]
    metadata: dict

base_panel = prices.set_index(["date", "symbol"]).sort_index()
close_values = base_panel["close"].to_numpy(dtype=float)
vector_ta_panel = base_panel.copy()
vector_ta_panel["vta__sma_20"] = vector_ta.sma(close_values, 20)
vector_ta_panel["vta__ema_20"] = vector_ta.ema(close_values, 20)
vector_ta_panel["vta__rsi_14"] = vector_ta.rsi(close_values, 14)
vector_ta_panel["vta__macd_12_26_9"] = vector_ta.macd(close_values, 12, 26, 9, "ema")[0]

vector_ta_contract = BuiltFeatureSet(
    family="vector_ta",
    df=vector_ta_panel,
    feature_cols=["vta__sma_20", "vta__ema_20", "vta__rsi_14", "vta__macd_12_26_9"],
    metadata={
        "library": "vector-ta",
        "library_version": md.version("vector-ta"),
        "vendor": "demo",
        "symbol_count": int(vector_ta_panel.index.get_level_values("symbol").nunique()),
        "cuda_symbol_count": int(vector_ta_status.loc[0, "cuda_symbol_count"]),
    },
)
built = vector_ta_contract
vector_ta_contract.df[vector_ta_contract.feature_cols].tail()


,,vta__sma_20,vta__ema_20,vta__rsi_14,vta__macd_12_26_9
date,symbol,,,,
2025-05-13,AAPL,69.1183,69.1327,56.7616,0.7551
2025-05-14,AAPL,69.2983,69.1884,55.0991,0.7111
2025-05-15,AAPL,69.5209,69.2384,55.0759,0.6683
2025-05-16,AAPL,69.7010,69.3079,56.4481,0.6475
2025-05-19,AAPL,69.8001,69.3661,56.0888,0.6198


In [4]:
built.df.tail()

,,open,high,low,close,volume,vta__sma_20,vta__ema_20,vta__rsi_14,vta__macd_12_26_9
date,symbol,,,,,,,,,
2025-05-13,AAPL,69.9203,70.4671,69.2820,69.9919,"3,265,126.0000",69.1183,69.1327,56.7616,0.7551
2025-05-14,AAPL,69.3637,70.5847,69.0344,69.7173,"2,688,905.0000",69.2983,69.1884,55.0991,0.7111
2025-05-15,AAPL,69.7905,70.9508,69.5032,69.7136,"3,593,169.0000",69.5209,69.2384,55.0759,0.6683
2025-05-16,AAPL,69.8128,71.2834,69.6055,69.9684,"4,028,724.0000",69.7010,69.3079,56.4481,0.6475
2025-05-19,AAPL,69.8496,70.4256,69.3163,69.9188,"3,691,978.0000",69.8001,69.3661,56.0888,0.6198


## Benchmarking Against Existing Modules

The reason to add VectorTA is speed and implementation diversity. It should be benchmarked against `compute_features_worldclass` and `pandas_ta_classic` for the same symbol universe and horizons.

In [5]:
from quant_warehouse.platforms.data_providers.fmp.feature_engineering import build_price_technical_features

base_features = build_price_technical_features("AAPL", prices)
comparison = pd.DataFrame([
    {"family": "worldclass_pandas", "rows": len(base_features.df), "features": len(base_features.feature_cols)},
    {"family": "vector_ta_contract", "rows": len(built.df), "features": len(built.feature_cols)},
])
comparison

,family,rows,features
0,worldclass_pandas,360,122
1,vector_ta_contract,360,4


## Target Engineering Connection

VectorTA features can be joined to forward-return or rank labels. Keep the library choice in feature metadata so future experiments can compare `vta__sma_20` versus `ta_overlap__sma_20` versus custom CUDA indicators.

In [6]:
from quant_warehouse.platforms.data_providers.fmp.target_engineering import build_forward_return_labels

labels = build_forward_return_labels(prices[["symbol", "date", "close"]], horizons=[5, 20])
joined = built.df.reset_index().merge(labels, on=["date", "symbol"], how="inner")
joined.tail()

,date,symbol,open,high,low,close,volume,vta__sma_20,vta__ema_20,vta__rsi_14,vta__macd_12_26_9,horizon,target_name,target_value
715,2025-05-15,AAPL,69.7905,70.9508,69.5032,69.7136,"3,593,169.0000",69.5209,69.2384,55.0759,0.6683,20,forward_return_20d,NaN
716,2025-05-16,AAPL,69.8128,71.2834,69.6055,69.9684,"4,028,724.0000",69.7010,69.3079,56.4481,0.6475,5,forward_return_5d,NaN
717,2025-05-16,AAPL,69.8128,71.2834,69.6055,69.9684,"4,028,724.0000",69.7010,69.3079,56.4481,0.6475,20,forward_return_20d,NaN
718,2025-05-19,AAPL,69.8496,70.4256,69.3163,69.9188,"3,691,978.0000",69.8001,69.3661,56.0888,0.6198,5,forward_return_5d,NaN
719,2025-05-19,AAPL,69.8496,70.4256,69.3163,69.9188,"3,691,978.0000",69.8001,69.3661,56.0888,0.6198,20,forward_return_20d,NaN


## Notes

- `vector-ta` is installed in this environment with CUDA symbols available.
- The adapter boundary lives under `platforms/computation_libraries/vector_ta/` and should expose reusable functions that return `BuiltFeatureSet`.
- Do not replace existing feature engineering modules; add VectorTA as a benchmarkable implementation choice.